# 实验 1 — merge disposition：第二次 ingest 会发生什么？

**目标**：观察 `write_disposition='merge'` 如何防止重复数据，并与 `replace` 对比。

**前提**：已运行过 `make ingest`（`data/sandbox.duckdb` 存在）。

In [ ]:
import duckdb
import subprocess

DB = "../data/sandbox.duckdb"

def count(table: str) -> int:
    return duckdb.connect(DB).execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]

print("=== 第一次 ingest 后 ===")
print(f"daily_rates   行数: {count('raw_ecb.daily_rates')}")
print(f"currencies_meta 行数: {count('raw_ecb.currencies_meta')}")

In [ ]:
# 再跑一次 ingest
result = subprocess.run(
    ["uv", "run", "python", "dlt_pipelines/ecb_rates.py"],
    capture_output=True, text=True, cwd=".."
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

In [ ]:
print("=== 第二次 ingest 后 ===")
print(f"daily_rates   行数: {count('raw_ecb.daily_rates')}  ← 和上面一样吗？")
print(f"currencies_meta 行数: {count('raw_ecb.currencies_meta')}  ← replace 每次都重写")

## 思考

- `daily_rates` 用了 `write_disposition='merge'`，以 `(date, currency)` 为 primary_key，所以重复跑不会产生重复行。
- `currencies_meta` 用了 `write_disposition='replace'`，每次都整表替换，但因为数据没变所以行数一样。
- 试一试：把 `ecb_rates.py` 里 `daily_rates` 的 `write_disposition` 改成 `'append'`，再跑两次，对比行数变化。

In [ ]:
# 查看 dlt 内部的 _dlt_loads 表 — 每次 load 都有一条记录
con = duckdb.connect(DB)
con.execute("SELECT * FROM raw_ecb._dlt_loads ORDER BY inserted_at DESC LIMIT 5").df()